# Cool Method - Fine-tune Your Own Reader (ModernBERT)  *(ADVANCED, optional)*

This is the **third reader**.

**Read this before running:** - This notebook assumes more comfort than any other in the course.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# --- Make your work survive a Colab reset (mount Drive; falls back to a local folder) ---
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode:", SMOKE)

## Install

ModernBERT needs a recent `transformers` (it landed in 4.48).

In [ ]:
%pip install -q -U "transformers>=4.48" "datasets>=2.19"

## Your labels become a training set

Train on the labels you already made: the hand-labeled gold set plus the AI labels from Week 7,
which you saved to your Drive project folder as `week07_labels.csv`.

In [ ]:
import os
import pandas as pd

labels_path = os.path.join(PROJECT_DIR, "week07_labels.csv")
if os.path.exists(labels_path):
    data = pd.read_csv(labels_path)
    text_col = "text"
    label_col = "gold" if "gold" in data.columns else "label"
    data = data[[text_col, label_col]].dropna().rename(columns={text_col: "text", label_col: "label"})
else:
    print("No week07_labels.csv found - using a tiny demo set. Run Week 7 first for the real thing.")
    data = pd.DataFrame({
        "text": ["a wonderful, moving film", "a boring, lifeless mess",
                 "gorgeous but hollow", "an instant classic", "a dreadful slog"],
        "label": ["positive", "negative", "mixed", "positive", "negative"],
    })
classes = sorted(data["label"].unique())
label2id = {c:i for i,c in enumerate(classes)}
data["y"] = data["label"].map(label2id)
print(data.head(), "\nclasses:", classes)

## Fine-tune

`RUN` is `False` by default so you can read the whole notebook safely (and so the offline test
harness does not try to download a model).

In [ ]:
RUN = False  # set True in Colab with a T4 GPU and your real labels

if RUN and not SMOKE:
    import numpy as np
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    MODEL = "answerdotai/ModernBERT-base"
    tok = AutoTokenizer.from_pretrained(MODEL)
    ds = Dataset.from_pandas(data[["text","y"]]).rename_column("y","labels")
    ds = ds.map(lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=128),
                batched=True)
    ds = ds.train_test_split(test_size=0.2, seed=0)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(classes), id2label={i:c for c,i in label2id.items()},
        label2id=label2id)

    out_dir = os.path.join(PROJECT_DIR, "modernbert-finetune")  # checkpoints to Drive
    args = TrainingArguments(output_dir=out_dir, num_train_epochs=3,
        per_device_train_batch_size=8, learning_rate=5e-5,
        eval_strategy="epoch", save_strategy="epoch", logging_steps=10,
        report_to="none")
    def metrics(p):
        preds = p.predictions.argmax(-1)
        return {"accuracy": float((preds == p.label_ids).mean())}
    trainer = Trainer(model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["test"], compute_metrics=metrics)
    trainer.train()
    trainer.save_model(os.path.join(out_dir, "final"))
    print("saved fine-tuned model ->", os.path.join(out_dir, "final"))
else:
    print("RUN is False (or SMOKE) - skipping training. Set RUN = True in Colab to fine-tune.")

## You own a reader now

When this finishes you have a model in your Drive folder that classifies *your* categories,
runs for free, and is far faster than prompting an API for the same narrow task.

**Sketch:** report your fine-tuned model's accuracy on a held-out slice, and one example it gets
right that Week 3's logistic regression got wrong - and say why the fine-tune could see it.